# VAW Phase 1 - Colab run: the final deliverable (scene graph stories, report page, annotation benchmark)
CPU is enough. Needs notebooks 05 (context features) and 07 (scene layout + depth); notebook 06 (evaluation) should be done so the page can show its findings.
Steps: (1) build the interaction scene graph and narrative of every clip, (2) render the showcase videos and the report page, (3) make the annotation sheets for the team,
(4) after the team has annotated, measure the detectors against human labels. Everything is explained in `docs/VERIFY_CONTEXT.md`.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!python -m pytest -q

## 1. Build the stories (scene graph episodes + narrative for every clip)
Prints how many clips show each predicate per category. Compare every category with Normal. Resumable (`--force` recomputes everything).

In [ ]:
!python -m src.context.story --config configs/colab.yaml

## 2. One story in words (change the clip id)
This is the 'context' the project is about: scene facts, then who does what to whom, when, with approximate meters, then a summary.

In [ ]:
!python -m src.context.story --config configs/colab.yaml --clips Stalking_v3 --example Stalking_v3 | tail -15

## 3. Render the report page (showcase = top-evidence clips AND random clips per category, so it is not cherry-picked)
Videos stop before the first physical-act cue. Writes `VAW_results/report/index.html`, `categories_story.csv`, `clips_story.csv`. Add `--set showcase_top=10 showcase_random=10` for more clips.

In [ ]:
!python -m src.report.story_report --config configs/colab.yaml --workers 2

## 4. Make the annotation sheets for the team (about 60 clips, 4 sheets)
Each team member gets one sheet (Drive -> right click -> Open with Google Sheets) and annotates **without looking at the system's output**: for each behavior write 1 (it happens before the
physical violence) or 0, and `act_start_s` = the second the physical act starts. Instructions are written next to the sheets.

In [ ]:
!python -m src.report.benchmark make --config configs/colab.yaml --per-category 10

## 5. After the team has filled the sheets: evaluate the detectors against humans
Prints, per behavior: how many clips annotators marked, how many the detectors found, precision, recall, F1, and the agreement between annotators (kappa), plus how accurate the
'physical act starts here' cue is compared with the annotated act start. These numbers, not the category rates, are the honest precision / recall of the system.

In [ ]:
!python -m src.report.benchmark evaluate --config configs/colab.yaml

## 6. Rebuild the page with the benchmark numbers included
Run step 3 again after step 5: the findings box then shows the benchmark table. To share: download `VAW_results/report` (right click -> Download), unzip, open `index.html`.

In [ ]:
!python -m src.report.story_report --config configs/colab.yaml --workers 2